# Noisy Pending-Aware qLogNEI Campaign

This notebook demonstrates the v2.2 qLogNEI workflow: accepted pending suggestions can remain in the CSV log and are accounted for as `X_pending`, while review-pending rows must be resolved before requesting more suggestions.

In [ ]:
from pathlib import Path
import math
import shutil

from bo_forge import CampaignSession

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "configs" / "18_noisy_pending_qlognei.yaml"
SEED_LOG_PATH = PROJECT_ROOT / "examples" / "18_noisy_pending_qlognei_campaign_log.csv"
WORKING_LOG_PATH = PROJECT_ROOT / "examples" / "18_noisy_pending_qlognei_working_log.csv"
LATEST_SUGGESTIONS_PATH = PROJECT_ROOT / "examples" / "18_noisy_pending_qlognei_latest_suggestions.csv"
REPORT_DIR = PROJECT_ROOT / "reports"
TARGET_OBSERVED_ROWS = 15

Copy the committed seed log before mutating the campaign. CSV logs are the source of truth, so the tutorial works against an ignored working copy.

In [ ]:
REPORT_DIR.mkdir(exist_ok=True)
shutil.copyfile(SEED_LOG_PATH, WORKING_LOG_PATH)
campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)
campaign.validate()
campaign.summary()
campaign.qlog_nei_summary()
campaign.next_action()

The deterministic objective below is local to the notebook. It gives each suggested row a repeatable observed value without adding any new BO behavior.

In [ ]:
def simulate_activity(row):
    loading = float(row["catalyst_loading"])
    temperature = float(row["reaction_temperature"])
    loading_term = 0.48 + 0.78 * loading - 0.95 * (loading - 0.62) ** 2
    temperature_term = -0.00007 * (temperature - 98.0) ** 2
    smooth_variation = 0.025 * math.sin(8.0 * loading + 0.035 * temperature)
    return round(loading_term + temperature_term + smooth_variation, 6)

Run one-at-a-time dry-run suggestions, append explicitly, and record observations. The first committed log already contains an accepted pending qLogNEI row, so the first model-based suggestion accounts for it as `X_pending`.

In [ ]:
while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    suggestions = campaign.suggest_next(batch_size=1)
    suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
    campaign.append_suggestions(suggestions)
    for row_id in suggestions["row_id"]:
        campaign.review_suggestion(row_id, "accept")
        row = campaign.df.loc[campaign.df["row_id"] == row_id].iloc[0]
        campaign.mark_observed(row_id, objective_value=simulate_activity(row))

campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)
campaign.qlog_nei_summary()

Export the standard report and qLogNEI diagnostics. The diagnostics summarize pending-state readiness; they do not fit an additional model.

In [ ]:
campaign.export_report(REPORT_DIR / "18_noisy_pending_qlognei_report.md")
campaign.plot_progress(save_path=REPORT_DIR / "18_noisy_pending_qlognei_progress.png")
campaign.plot_diagnostics(save_path=REPORT_DIR / "18_noisy_pending_qlognei_diagnostics.png")
campaign.plot_qlog_nei_diagnostics(save_path=REPORT_DIR / "18_noisy_pending_qlognei_pending.png")